In [1]:
from datasets import load_dataset

dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1"
)

print(dataset)

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

c:\Users\User\NUS\DSA NLP\DSA4213_Assignment1\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\datasets--Salesforce--wikitext. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\User\NUS\DSA NLP\DSA4213_Assignment1\.venv\Lib\site-packages\huggingface_hub\file_downl

test-00000-of-00001.parquet:   0%|          | 0.00/733k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


train-00000-of-00001.parquet:   0%|          | 0.00/6.36M [00:00<?, ?B/s]

...

validation-00000-of-00001.parquet:   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 36718
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


In [2]:
print("First 10 rows:\n")

for i in range(10):
    print(f"Row {i}: {repr(dataset['train'][i]['text'])}")

First 10 rows:

Row 0: ''
Row 1: ' = Valkyria Chronicles III = \n'
Row 2: ''
Row 3: ' Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n'
Row 4: " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple ad

In [3]:
def preprocess_text(text):
    text = text.strip().lower()

    if not text:
        return []

    tokens = text.split()
    return tokens


train_tokens = []
val_tokens = []
test_tokens = []

for row in dataset["train"]:
    train_tokens.extend(preprocess_text(row["text"]))

for row in dataset["validation"]:
    val_tokens.extend(preprocess_text(row["text"]))

for row in dataset["test"]:
    test_tokens.extend(preprocess_text(row["text"]))


print("Training tokens:", len(train_tokens))
print("Validation tokens:", len(val_tokens))
print("Test tokens:", len(test_tokens))

print("\nFirst 30 tokens:")
print(train_tokens[:30])

Training tokens: 2051910
Validation tokens: 213886
Test tokens: 241211

First 30 tokens:
['=', 'valkyria', 'chronicles', 'iii', '=', 'senjō', 'no', 'valkyria', '3', ':', 'unrecorded', 'chronicles', '(', 'japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'valkyria', 'of', 'the', 'battlefield', '3', ')', ',', 'commonly', 'referred', 'to', 'as']


In [4]:
from collections import Counter

word_counts = Counter(train_tokens)

print("Number of unique words:", len(word_counts))

print("\n20 most common words:")
for word, count in word_counts.most_common(20):
    print(f"{word:15} {count}")

Number of unique words: 66649

20 most common words:
the             130768
,               99913
.               73388
of              57032
and             50735
in              44982
to              39522
a               36156
=               29570
"               28309
was             21008
@-@             16906
on              15141
as              15058
that            14351
's              14002
for             13794
with            13012
by              12718
)               12004


In [5]:
VOCAB_SIZE = 10000

special_tokens = ["<pad>", "<unk>"]

most_common_words = [
    word for word, count in word_counts.most_common()
    if word not in special_tokens
][:VOCAB_SIZE - len(special_tokens)]

vocab = special_tokens + most_common_words

word_to_id = {
    word: idx for idx, word in enumerate(vocab)
}

id_to_word = {
    idx: word for word, idx in word_to_id.items()
}

PAD_ID = word_to_id["<pad>"]
UNK_ID = word_to_id["<unk>"]

print("Vocabulary size:", len(vocab))
print("Padding ID:", PAD_ID)
print("Unknown ID:", UNK_ID)

print("\nFirst 20 vocabulary entries:")
print(vocab[:20])

Vocabulary size: 10000
Padding ID: 0
Unknown ID: 1

First 20 vocabulary entries:
['<pad>', '<unk>', 'the', ',', '.', 'of', 'and', 'in', 'to', 'a', '=', '"', 'was', '@-@', 'on', 'as', 'that', "'s", 'for', 'with']


In [6]:
def encode_tokens(tokens):
    return [
        word_to_id.get(token, UNK_ID)
        for token in tokens
    ]


train_ids = encode_tokens(train_tokens)
val_ids = encode_tokens(val_tokens)
test_ids = encode_tokens(test_tokens)

print("Original tokens:")
print(train_tokens[:20])

print("\nEncoded IDs:")
print(train_ids[:20])

print("\nDecoded tokens:")
print([id_to_word[idx] for idx in train_ids[:20]])

print("\nUnknown token percentage:")
for name, ids in [
    ("Train", train_ids),
    ("Validation", val_ids),
    ("Test", test_ids)
]:
    unk_pct = 100 * ids.count(UNK_ID) / len(ids)
    print(f"{name}: {unk_pct:.2f}%")

Original tokens:
['=', 'valkyria', 'chronicles', 'iii', '=', 'senjō', 'no', 'valkyria', '3', ':', 'unrecorded', 'chronicles', '(', 'japanese', ':', '戦場のヴァルキュリア3', ',', 'lit', '.', 'valkyria']

Encoded IDs:
[10, 3834, 3897, 885, 10, 1, 108, 3834, 92, 46, 1, 3897, 22, 784, 46, 1, 3, 6470, 4, 3834]

Decoded tokens:
['=', 'valkyria', 'chronicles', 'iii', '=', '<unk>', 'no', 'valkyria', '3', ':', '<unk>', 'chronicles', '(', 'japanese', ':', '<unk>', ',', 'lit', '.', 'valkyria']

Unknown token percentage:
Train: 8.02%
Validation: 9.75%
Test: 10.64%


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

CONTEXT_LENGTH = 32

class LanguageModelDataset(Dataset):
    def __init__(self, token_ids, context_length=32):
        self.tokens = torch.tensor(token_ids, dtype=torch.long)
        self.context_length = context_length

    def __len__(self):
        return (len(self.tokens) - 1) // self.context_length

    def __getitem__(self, idx):
        start = idx * self.context_length
        end = start + self.context_length

        x = self.tokens[start:end]
        y = self.tokens[start + 1:end + 1]

        return x, y

In [8]:
BATCH_SIZE = 32

train_dataset = LanguageModelDataset(
    train_ids, CONTEXT_LENGTH
)

val_dataset = LanguageModelDataset(
    val_ids, CONTEXT_LENGTH
)

test_dataset = LanguageModelDataset(
    test_ids, CONTEXT_LENGTH
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))
print("Test sequences:", len(test_dataset))

Training sequences: 64122
Validation sequences: 6683
Test sequences: 7537


In [9]:
x_batch, y_batch = next(iter(train_loader))

print("Input shape:", x_batch.shape)
print("Target shape:", y_batch.shape)

print("\nFirst input sequence:")
print(x_batch[0][:10].tolist())

print("\nFirst target sequence:")
print(y_batch[0][:10].tolist())

print("\nDecoded input:")
print([id_to_word[i] for i in x_batch[0][:10].tolist()])

print("\nDecoded target:")
print([id_to_word[i] for i in y_batch[0][:10].tolist()])

assert x_batch.shape == y_batch.shape
assert torch.equal(x_batch[:, 1:], y_batch[:, :-1])

print("\nInput-target alignment verified!")

Input shape: torch.Size([32, 32])
Target shape: torch.Size([32, 32])

First input sequence:
[18, 475, 13, 1476, 324, 31, 1, 2, 1205, 5]

First target sequence:
[475, 13, 1476, 324, 31, 1, 2, 1205, 5, 44]

Decoded input:
['for', 'full', '@-@', 'scale', 'production', 'had', '<unk>', 'the', 'initial', 'of']

Decoded target:
['full', '@-@', 'scale', 'production', 'had', '<unk>', 'the', 'initial', 'of', 'its']

Input-target alignment verified!


In [10]:
import json
import torch
from pathlib import Path

# Notebook is inside the notebooks/ folder,
# but VS Code may run it from the project root.
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(exist_ok=True)

# Save vocabulary
with open(DATA_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(word_to_id, f, ensure_ascii=False, indent=2)

# Save encoded datasets
torch.save({
    "train": train_ids,
    "validation": val_ids,
    "test": test_ids
}, DATA_DIR / "token_ids.pt")

# Save configuration
config = {
    "dataset": "Salesforce/wikitext",
    "subset": "wikitext-2-raw-v1",
    "vocab_size": len(vocab),
    "context_length": CONTEXT_LENGTH,
    "batch_size": BATCH_SIZE,
    "lowercase": True,
    "tokenizer": "whitespace",
    "pad_id": PAD_ID,
    "unk_id": UNK_ID
}

with open(DATA_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Preprocessing files saved successfully!")
print("Location:", DATA_DIR)

Preprocessing files saved successfully!
Location: c:\Users\User\NUS\DSA NLP\DSA4213_Assignment1\data
